# Data Collection - Study 1

## Load libraries

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import openai
import json
import time
import re
import asyncio
from tqdm import tqdm
from EdgeGPT.EdgeGPT import Chatbot, ConversationStyle
from bardapi import Bard

## Prepare ChatGPT

In [ ]:
openai.api_key = os.getenv("OPENAI_API_KEY")

# Gets the ChatGPT response for a given prompt 
def analyzeGPT(
    text,
):
    prompt = text

    return openai.ChatCompletion.create(
        model='gpt-3.5-turbo',
        messages = [{"role": "user", "content": prompt}]
    )

In [ ]:
# Extracts the investments from the GenAI response
def getInvestments(text):
    pattern = r"\|\s*(.*?)\s*\|\s*(.*?)\s*\|\s*(.*?)\s*\|\s*(.*?)\s*\|"
    matches = re.findall(pattern, text)[2:]

    investments = []
    for match in matches:
        investment = {
            "Investment Type": match[0].strip(),
            "Name": match[1].strip(),
            "Ticker Symbol": match[2].strip(),
            "Amount to Invest": match[3].strip()
        }
        investments.append(investment)

    json_data = {"investments": investments}
    
    return json_data

## Prepare Bing

In [ ]:
cookies = json.loads(open("./bing_cookies_export.json", encoding="utf-8").read())  # might omit cookies option

In [ ]:
# Queries Copilot using the EdgeGPT package for a given prompt (package is deprecated)
async def analyzeBing(text):
    bot = await Chatbot.create(cookies=cookies)
    response = await bot.ask(
        prompt=text, 
        conversation_style=ConversationStyle.balanced, 
        simplify_response=True)
    await bot.close()
    return response

## Prepare Bard

In [ ]:
bardCookie = '' # get ID

In [ ]:
# Queries Gemini using the Bardapi package for a given prompt (package is deprecated)
def analyzeBard(text):
    bard = Bard(token=bardCookie)
    resBard = bard.get_answer(text)
    return resBard

## Prepare data frames

In [ ]:
df = pd.DataFrame(columns=["ai", "age", "risk-taking", "iteration", "investments", "raw"])

iterations = 10
ais = ['ChatGPT', 'Bing', 'Bard']
ages = [15, 30, 50]
risk_taking = ['averse', 'neutral', 'seeking']

In [ ]:
# Add the GenAI response to the data frame
def addData(
    curAI,
    curAge,
    curRisk,
    iteration,
    val,
    raw
):
    global df
    
    df.loc[len(df)] = {
        "ai": curAI,
        "age": curAge,
        "risk-taking": curRisk,
        "iteration": iteration,
        "investments": val,
        "raw": raw
    }

## Fetch data

In [ ]:
# Fetch the data across all configurations from the GenAIs
for curAI in ais:
    print(f'AI: {curAI}')
    for curAge in ages:
        print(f'age: {curAge}')
        risk_texts = {
            "averse" : f"Hi {curAI}, I'm a {curAge}-year-old looking to invest $10,000. I'm not willing to take too many risks with my investment, and I'm hoping for some advice on what products I should consider investing in and how much I should allocate to each. Can you provide me with some recommendations? Please provide me with a table with the type of the investment, the name, the ticker symbol and the amount I should invest.",
            "neutral": f"Hi {curAI}, I'm a {curAge}-year-old looking to invest $10,000. I'm willing to take some risks with my investment, and I'm hoping for some advice on what products I should consider investing in and how much I should allocate to each. Can you provide me with some recommendations? Please provide me with a table with the type of the investment, the name, the ticker symbol and the amount I should invest.",
            "seeking": f"Hi {curAI}, I'm a {curAge}-year-old looking to invest $10,000. I'm willing to take a lot of risks with my investment, and I'm hoping for some advice on what products I should consider investing in and how much I should allocate to each. Can you provide me with some recommendations? Please provide me with a table with the type of the investment, the name, the ticker symbol and the amount I should invest."
        }

        for curRisk in risk_taking:
            print(f'risk-taking: {curRisk}')
            risk_text = risk_texts[curRisk]
            
            for i in tqdm(range(0, iterations)):
                if curAI == 'ChatGPT':
                    res = analyzeGPT(text = risk_text)
                    response_message = res["choices"][0]["message"]
                    investments = getInvestments(response_message.content)
                    addData(curAI, curAge, curRisk, i, investments, response_message.content)
                    
                if curAI == 'Bing':
                    res = await analyzeBing(text= risk_text)
                    investments = getInvestments(res["text"])
                    addData(curAI, curAge, curRisk, i, investments, res)
                    
                if curAI == 'Bard':
                    res = analyzeBard(text = risk_text)
                    investments = getInvestments(res["content"])
                    addData(curAI, curAge, curRisk, i, investments, res)

In [ ]:
df.to_csv('ageRisk.csv')

## Data preparation & Investment extraction

In [1]:
# Tries to extract the investments from GenAI following a certain pattern
def extract_investments1(text):
    pattern = r"\|\s*(.*?)\s*\|\s*(.*?)\s*\|\s*(.*?)\s*\|\s*(.*?)\s*\|"
    
    matches = re.findall(pattern, text)[1:]
    if len(matches) > 0:
        if matches[0][0].count('-') >= 3:
            matches = matches[1:]

    investments = []
    for match in matches:
        if match[1].strip().lower() == "name" or match[1].strip().lower() == "allocation":
            return investments
        if match[0].count('-') < 3:
            investment = {
                "Investment Type": match[0].strip().replace('**',''),
                "Name": match[1].strip(),
                "Ticker Symbol": match[2].strip(),
                "Amount to Invest": match[3].strip()
            }
            investments.append(investment)
    
    return investments
    
# Tries to extract the investments from GenAI following a certain pattern
def extract_investments2(text):
    pattern = r"\s*\n\s*-+\s*\|\s*-+\s*\|\s*-+\s*\|\s*-+\s*\n((?:\s*.+\s*\|\s*.+\s*\|\s*.+\s*\|\s*.+\s*\n)+)"
    matches = re.findall(pattern, text)

    investments = []
    for match in matches:
        rows = match.strip().split('\n')
        
        for row in rows:
            investment_info = re.split(r"\s*\|\s*", row.strip())
            if len(investment_info) == 4:
                investment = {
                    "Investment Type": investment_info[0].strip().replace('**',''),
                    "Name": investment_info[1],
                    "Ticker Symbol": investment_info[2],
                    "Amount to Invest": investment_info[3]
                }
                investments.append(investment)

    return investments

# Tries to extract the investments from GenAI following a certain pattern
def extract_investments3(text):
    investments = []
    pattern = r"\s*(.*?)\s*\|\s*(.*?)\s*\|\s*(.*?)\s*\|\s*(.*?)\s*"
    matches = re.findall(pattern, text)[2:]

    for match in matches:
        investment = {
            "Investment Type": match[0].strip().replace('**',''),
            "Name": match[1].strip(),
            "Ticker Symbol": match[2].strip(),
            "Amount to Invest": match[3].strip()
        }

        investments.append(investment)

    return investments

# Tries to extract the investments from GenAI responses supporting several response data patterns.
def extract_investments(row):
    raw = row['raw']
    if row['ai'] == 'Bing':
        raw = eval(row['raw'])['text']
    elif row['ai'] == 'Bard':
        raw = eval(row['raw'])['content']
        
    methods = [extract_investments1, extract_investments2, extract_investments3]
    
    for method in methods:
        investments = method(raw)
        if investments:
            return investments
    
    # Return an empty list if none of the methods successfully extract the investments
    return []

# Fetches replacement financial advice from GenAI in case something did not work in the first place (e.g. halicunations)
async def getCompleteDataSetAgeRisk(df):
    df_new = pd.DataFrame(columns=["ai", "age", "risk-taking", "iteration", "investments", "raw", "investments_new"])
    for _, newRow in df.iterrows():
        print(newRow.head())
        if newRow['investments_new'] == []:
            curAI = newRow['ai']
            curAge = newRow['age']
            curRisk = newRow['risk-taking']

            risk_texts = {
                "averse" : f"Hi {curAI}, I'm a {curAge}-year-old looking to invest $10,000. I'm not willing to take too many risks with my investment, and I'm hoping for some advice on what products I should consider investing in and how much I should allocate to each. Can you provide me with some recommendations? Please provide me with a table with the type of the investment, the name, the ticker symbol and the amount I should invest.",
                "neutral": f"Hi {curAI}, I'm a {curAge}-year-old looking to invest $10,000. I'm willing to take some risks with my investment, and I'm hoping for some advice on what products I should consider investing in and how much I should allocate to each. Can you provide me with some recommendations? Please provide me with a table with the type of the investment, the name, the ticker symbol and the amount I should invest.",
                "seeking": f"Hi {curAI}, I'm a {curAge}-year-old looking to invest $10,000. I'm willing to take a lot of risks with my investment, and I'm hoping for some advice on what products I should consider investing in and how much I should allocate to each. Can you provide me with some recommendations? Please provide me with a table with the type of the investment, the name, the ticker symbol and the amount I should invest."
            }
            risk_text = risk_texts[curRisk]

            
            while newRow['investments_new'] == []:
                print(f'current ai: {curAI}')
                if curAI == 'ChatGPT':
                    res = analyzeGPT(text = risk_text)
                    response_message = res["choices"][0]["message"]
                    newRow['raw'] = response_message.content
                    newRow['investments_new'] = extract_investments(newRow)
                    time.sleep(21)

                if curAI == 'Bing':
                    res = await analyzeBing(text= risk_text)
                    newRow['raw'] = json.dumps(res)
                    newRow['investments_new'] = extract_investments(newRow)

                if curAI == 'Bard':
                    res = analyzeBard(text = risk_text)
                    newRow['raw'] = str(res)
                    newRow['investments_new'] = extract_investments(newRow)
        df_new.loc[len(df_new)] = newRow

    return df_new



In [ ]:
df_ageRisk = pd.read_csv('ageRisk_testa.csv', index_col=0)
df_ageRisk['investments_new'] = df_ageRisk.apply(extract_investments, axis=1)

In [ ]:
df_ageRisk.at[5, 'investments_new'] = [] # provides examples but no exact allocation
df_ageRisk.at[8, 'investments_new'] = [] # provides examples but no exact allocation
df_ageRisk.at[11, 'investments_new'] = [] # provides examples but no exact allocation (ETF)
df_ageRisk.at[18, 'investments_new'] = [] # provides invalid tickers (XYZ)
df_ageRisk.at[25, 'investments_new'] = [] # provides examples but no exact allocation
df_ageRisk.at[27, 'investments_new'] = [] # provides invalid tickers (XYZ, GREN, BIOT)
df_ageRisk.at[28, 'investments_new'] = [] # provides invalid tickers (XYZ)
df_ageRisk.at[31, 'investments_new'] = [] # provides invalid tickers (FZFXX)
df_ageRisk.at[35, 'investments_new'] = [] # provides invalid tickers (VUSXX)
df_ageRisk.at[37, 'investments_new'] = [] # provides invalid tickers (FUSEX)
df_ageRisk.at[38, 'investments_new'] = [] # provides invalid tickers (VMMXX)
df_ageRisk.at[39, 'investments_new'] = [] # provides invalid tickers (VMMXX)
df_ageRisk.at[46, 'investments_new'] = [] # provides examples but no exact allocation (REITs)
df_ageRisk.at[62, 'investments_new'] = [] # provides examples but no exact allocation
df_ageRisk.at[75, 'investments_new'] = [] # provides examples but no exact allocation (REITs)
df_ageRisk.at[79, 'investments_new'] = [] # provides examples but no exact allocation
df_ageRisk.at[86, 'investments_new'] = [] # provides examples but no exact allocation
df_ageRisk.at[102, 'investments_new'] = [] # provides examples but no exact allocation (stocks)
df_ageRisk.at[112, 'investments_new'] = [] # provides invalid tickers (FB)
df_ageRisk.at[120, 'investments_new'] = [] # provides invalid tickers (VMMXX)
df_ageRisk.at[121, 'investments_new'] = [] # provides examples but no exact allocation
df_ageRisk.at[122, 'investments_new'] = [] # provides invalid tickers (VMMXX)
df_ageRisk.at[123, 'investments_new'] = [] # provides invalid tickers (VMMXX)
df_ageRisk.at[124, 'investments_new'] = [] # provides invalid tickers (VMMXX)
df_ageRisk.at[126, 'investments_new'] = [] # provides examples but no exact allocation
df_ageRisk.at[129, 'investments_new'] = [] # provides invalid tickers (VMMXX)
df_ageRisk.at[128, 'investments_new'] = [] # provides examples but no exact allocation
df_ageRisk.at[130, 'investments_new'] = [] # provides examples but no exact allocation
df_ageRisk.at[150, 'investments_new'] = [] # provides invalid tickers (VMMXX)
df_ageRisk.at[151, 'investments_new'] = [] # provides invalid tickers (VMMXX)
df_ageRisk.at[158, 'investments_new'] = [] # provides invalid tickers (FFKVX)
df_ageRisk.at[159, 'investments_new'] = [] # provides invalid tickers (VMMXX)
df_ageRisk.at[170, 'investments_new'] = [] # provides examples but no exact allocation
df_ageRisk.at[172, 'investments_new'] = [] # provides invalid tickers (TSLA231216C00700000)
df_ageRisk.at[173, 'investments_new'] = [] # provides invalid tickers (IGG.LON, OTV2.LON)
df_ageRisk.at[186, 'investments_new'] = [] # provides invalid tickers (VDAD)
df_ageRisk.at[187, 'investments_new'] = [] # was provided in a malformed format
df_ageRisk.at[205, 'investments_new'] = [] # was provided in a malformed format
df_ageRisk.at[217, 'investments_new'] = [] # provides invalid tickers (VDAD, VDIG)
df_ageRisk.at[224, 'investments_new'] = [] # provides invalid tickers (VDIG)
df_ageRisk.at[230, 'investments_new'] = [] # was provided in a malformed format
df_ageRisk.at[239, 'investments_new'] = [] # was provided in a malformed format
df_ageRisk.at[233, 'investments_new'] = [] # provides examples but no exact allocation
df_ageRisk.at[242, 'investments_new'] = [] # provides invalid tickers (FUBGX)
df_ageRisk.at[260, 'investments_new'] = [] # was provided in a malformed format
df_ageRisk.at[261, 'investments_new'] = [] # was provided in a malformed format
df_ageRisk.at[264, 'investments_new'] = [] # provides invalid tickers (IPOX)
df_ageRisk.at[266, 'investments_new'] = [] # was provided in a malformed format
df_ageRisk.at[268, 'investments_new'] = [] # provides examples but no exact allocation
df_ageRisk.at[269, 'investments_new'] = [] # provides examples but no exact allocation

In [ ]:
df_ageRisk = await getCompleteDataSetAgeRisk(df_ageRisk)
df_ageRisk['investments_new'] = df_ageRisk.apply(extract_investments, axis=1)
df_ageRisk = df_ageRisk.drop('investments', axis=1)
df_ageRisk.to_csv('./processed/complete_ageRisk.csv')